In [ ]:
import pickle
import time
import numpy as np
from joblib import Parallel, delayed
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsClassifier
from sgw_numpy import sgw_cpu
import torch
from risgw import risgw_gpu
from SEINT_numpy import SEINT
import ot

with open('modelnet40_rotated.pkl', 'rb') as f:
    data = pickle.load(f)
    train_3D_small = data['train_3D_small']
    train_label_small = data['train_label_small']

K = 5
n_neighbors = 5
seed = 42
N = len(train_label_small)

### RISGW

In [ ]:
risgw_rep = 50             
t_all = time.time()

def compute_pair(i, j):
    A_t = torch.from_numpy(np.asarray(train_3D_small[i], dtype=np.float32))
    B_t = torch.from_numpy(np.asarray(train_3D_small[j], dtype=np.float32))
    dist = risgw_gpu(A_t, B_t,'cpu', risgw_rep)
    return i, j, float(dist)


pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

print(f"[INFO] Start precomputing full {N}x{N} RISGW distance matrix with {len(pairs)} pairs ...")
t0 = time.time()
results = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_pair)(i, j) for i, j in pairs
)

risgw_dist_full = np.zeros((N, N), dtype=np.float64)
for i, j, dist in results:
    risgw_dist_full[i, j] = dist
    risgw_dist_full[j, i] = dist
print(f"[INFO] Done precomputing. Time: {time.time() - t0:.2f}s")

kf = KFold(n_splits=K, shuffle=True, random_state=42)

acc_list = []
fold_times = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
    t_fold = time.time()
    print(f"\n===== Fold {fold}/{K} =====")

    y_train = train_label_small[train_idx]
    y_val = train_label_small[val_idx]

    D_train = risgw_dist_full[np.ix_(train_idx, train_idx)]
    D_val = risgw_dist_full[np.ix_(val_idx, train_idx)]

    clf = KNeighborsClassifier(metric='precomputed', n_neighbors=n_neighbors)
    clf.fit(D_train, y_train)

    y_pred = clf.predict(D_val)
    acc = (y_pred == y_val).mean() * 100
    acc_list.append(acc)

    fold_time = time.time() - t_fold
    fold_times.append(fold_time)

    print(f"Fold {fold} Val Accuracy: {acc:.2f}% | Time: {fold_time:.2f}s")

print("\n===== Final CV Result =====")
print(f"Avg Acc: {np.mean(acc_list):.2f}% ± {np.std(acc_list):.2f}%")
print(f"Avg Fold Time: {np.mean(fold_times):.2f}s | Total Time: {time.time() - t_all:.2f}s")


### SGW

In [ ]:
sgw_rep = 50  

t_all = time.time()

def compute_pair(i, j):
    dist = sgw_cpu(train_3D_small[i], train_3D_small[j], sgw_rep)
    return i, j, dist


pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

print(f"[INFO] Start precomputing full {N}x{N} SGW distance matrix with {len(pairs)} pairs ...")
t0 = time.time()
results = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_pair)(i, j) for i, j in pairs
)

sgw_dist_full = np.zeros((N, N), dtype=np.float64)
for i, j, dist in results:
    sgw_dist_full[i, j] = dist
    sgw_dist_full[j, i] = dist
print(f"[INFO] Done precomputing. Time: {time.time() - t0:.2f}s")


kf = KFold(n_splits=K, shuffle=True, random_state=42)

acc_list = []
fold_times = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
    t_fold = time.time()
    print(f"\n===== Fold {fold}/{K} =====")

    y_train = train_label_small[train_idx]
    y_val = train_label_small[val_idx]

    D_train = sgw_dist_full[np.ix_(train_idx, train_idx)]
    D_val = sgw_dist_full[np.ix_(val_idx, train_idx)]

    clf = KNeighborsClassifier(metric='precomputed', n_neighbors=n_neighbors)
    clf.fit(D_train, y_train)

    y_pred = clf.predict(D_val)
    acc = (y_pred == y_val).mean() * 100
    acc_list.append(acc)

    fold_time = time.time() - t_fold
    fold_times.append(fold_time)

    print(f"Fold {fold} Val Accuracy: {acc:.2f}% | Time: {fold_time:.2f}s")

print("\n===== Final CV Result =====")
print(f"Avg Acc: {np.mean(acc_list):.2f}% ± {np.std(acc_list):.2f}%")
print(f"Avg Fold Time: {np.mean(fold_times):.2f}s | Total Time: {time.time() - t_all:.2f}s")


### SEINT

In [ ]:
riot_kwargs = dict(    
        rd = None,
        initial = True,
        lp = 2,
        rep = 50,
        maxed = True,
        determin = False,
        set_seed = 42,
        rd_rad = 2,
        acc = True
)

t_all = time.time()

def compute_pair(i, j):
    dist = SEINT(train_3D_small[i], train_3D_small[j], **riot_kwargs)
    return i, j, dist

pairs = [(i, j) for i in range(N) for j in range(i+1, N)]

print(f"[INFO] Start precomputing full {N}x{N} distance matrix with {len(pairs)} pairs ...")
t0 = time.time()
results = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_pair)(i, j) for i, j in pairs
)
rot_dist_full = np.zeros((N, N), dtype=np.float64)
for i, j, dist in results:
    rot_dist_full[i, j] = dist
    rot_dist_full[j, i] = dist
print(f"[INFO] Done precomputing. Time: {time.time() - t0:.2f}s")

with open('rot_dist_full.pkl', 'rb') as f:
    data = pickle.load(f)
    rot_dist_full = data['rot_dist_full']

kf = KFold(n_splits=K, shuffle=True, random_state=42)

acc_list = []
fold_times = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
    t_fold = time.time()
    print(f"\n===== Fold {fold}/{K} =====")

    y_train = train_label_small[train_idx]
    y_val = train_label_small[val_idx]

    D_train = rot_dist_full[np.ix_(train_idx, train_idx)]
    D_val = rot_dist_full[np.ix_(val_idx, train_idx)]

    clf = KNeighborsClassifier(metric='precomputed', n_neighbors=n_neighbors)
    clf.fit(D_train, y_train)

    y_pred = clf.predict(D_val)
    acc = (y_pred == y_val).mean() * 100
    acc_list.append(acc)

    fold_time = time.time() - t_fold
    fold_times.append(fold_time)

    print(f"Fold {fold} Val Accuracy: {acc:.2f}% | Time: {fold_time:.2f}s")

print("\n===== Final CV Result =====")
print(f"Avg Acc: {np.mean(acc_list):.2f}% ± {np.std(acc_list):.2f}%")
print(f"Avg Fold Time: {np.mean(fold_times):.2f}s | Total Time: {time.time() - t_all:.2f}s")


In [ ]:
iseint_kwargs = dict(    
        rd = None,
        initial = True,
        lp = 2,
        rep = 50,
        maxed = False,
        determin = False,
        set_seed = 42,
        rd_rad = 2,
        acc = True
)


N = len(train_label_small)

t_all = time.time()

def compute_pair(i, j):
    dist = SEINT(train_3D_small[i], train_3D_small[j], **iseint_kwargs)
    return i, j, dist

pairs = [(i, j) for i in range(N) for j in range(i+1, N)]

print(f"[INFO] Start precomputing full {N}x{N} distance matrix with {len(pairs)} pairs ...")
t0 = time.time()
results = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_pair)(i, j) for i, j in pairs
)
iseint_dist_full = np.zeros((N, N), dtype=np.float64)
for i, j, dist in results:
    iseint_dist_full[i, j] = dist
    iseint_dist_full[j, i] = dist
print(f"[INFO] Done precomputing. Time: {time.time() - t0:.2f}s")

kf = KFold(n_splits=K, shuffle=True, random_state=42)

acc_list = []
fold_times = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
    t_fold = time.time()
    print(f"\n===== Fold {fold}/{K} =====")

    y_train = train_label_small[train_idx]
    y_val = train_label_small[val_idx]

    D_train = iseint_dist_full[np.ix_(train_idx, train_idx)]
    D_val = iseint_dist_full[np.ix_(val_idx, train_idx)]

    clf = KNeighborsClassifier(metric='precomputed', n_neighbors=n_neighbors)
    clf.fit(D_train, y_train)

    y_pred = clf.predict(D_val)
    acc = (y_pred == y_val).mean() * 100
    acc_list.append(acc)

    fold_time = time.time() - t_fold
    fold_times.append(fold_time)

    print(f"Fold {fold} Val Accuracy: {acc:.2f}% | Time: {fold_time:.2f}s")

print("\n===== Final CV Result =====")
print(f"Avg Acc: {np.mean(acc_list):.2f}% ± {np.std(acc_list):.2f}%")
print(f"Avg Fold Time: {np.mean(fold_times):.2f}s | Total Time: {time.time() - t_all:.2f}s")

### W_2

In [ ]:
use_sinkhorn = False      
sinkhorn_reg = 1e-2      


t_all = time.time()

def w2_distance(Xi, Xj):
    Xi = np.asarray(Xi, dtype=float)
    Xj = np.asarray(Xj, dtype=float)
    n, m = len(Xi), len(Xj)
    a = np.ones((n,)) / n
    b = np.ones((m,)) / m
    M = ot.dist(Xi, Xj, metric='euclidean') ** 2  # (n, m)

    cost = ot.emd2(a, b, M)
    return float(np.sqrt(cost))

def compute_pair(i, j):
    dist = w2_distance(train_3D_small[i], train_3D_small[j])
    return i, j, dist

pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

solver_name = "sinkhorn" if use_sinkhorn else "emd"
print(f"[INFO] Start precomputing full {N}x{N} W2 distance matrix ({solver_name}) with {len(pairs)} pairs ...")
t0 = time.time()
results = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_pair)(i, j) for i, j in pairs
)

w2_dist_full = np.zeros((N, N), dtype=np.float64)
for i, j, dist in results:
    w2_dist_full[i, j] = dist
    w2_dist_full[j, i] = dist
print(f"[INFO] Done precomputing. Time: {time.time() - t0:.2f}s")

kf = KFold(n_splits=K, shuffle=True, random_state=42)
acc_list, fold_times = [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
    t_fold = time.time()
    print(f"\n===== Fold {fold}/{K} =====")

    y_train = train_label_small[train_idx]
    y_val = train_label_small[val_idx]

    D_train = w2_dist_full[np.ix_(train_idx, train_idx)]
    D_val = w2_dist_full[np.ix_(val_idx, train_idx)]

    clf = KNeighborsClassifier(metric='precomputed', n_neighbors=n_neighbors)
    clf.fit(D_train, y_train)

    y_pred = clf.predict(D_val)
    acc = (y_pred == y_val).mean() * 100
    acc_list.append(acc)

    fold_time = time.time() - t_fold
    fold_times.append(fold_time)
    print(f"Fold {fold} Val Accuracy: {acc:.2f}% | Time: {fold_time:.2f}s")

print("\n===== Final CV Result (W2) =====")
print(f"Avg Acc: {np.mean(acc_list):.2f}% ± {np.std(acc_list):.2f}%")
print(f"Avg Fold Time: {np.mean(fold_times):.2f}s | Total Time: {time.time() - t_all:.2f}s")



### SW

In [ ]:
from ot import sliced as ots
num_projections = 50   


t_all = time.time()

def compute_pair(i, j):
    Xi = np.asarray(train_3D_small[i], dtype=float)
    Xj = np.asarray(train_3D_small[j], dtype=float)
    dist = ots.sliced_wasserstein_distance(
        Xi, Xj, n_projections=num_projections, seed=seed
    )
    return i, j, float(dist)

pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

print(f"[INFO] Start precomputing full {N}x{N} SW distance matrix with {len(pairs)} pairs ...")
t0 = time.time()
results = Parallel(n_jobs=-1, backend='loky', verbose=1)(
    delayed(compute_pair)(i, j) for i, j in pairs
)
sw_dist_full = np.zeros((N, N), dtype=np.float64)
for i, j, dist in results:
    sw_dist_full[i, j] = dist
    sw_dist_full[j, i] = dist
print(f"[INFO] Done precomputing. Time: {time.time() - t0:.2f}s")

kf = KFold(n_splits=K, shuffle=True, random_state=42)
acc_list, fold_times = [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(N)), start=1):
    t_fold = time.time()
    print(f"\n===== Fold {fold}/{K} =====")

    y_train = train_label_small[train_idx]
    y_val = train_label_small[val_idx]

    D_train = sw_dist_full[np.ix_(train_idx, train_idx)]
    D_val = sw_dist_full[np.ix_(val_idx, train_idx)]

    clf = KNeighborsClassifier(metric='precomputed', n_neighbors=n_neighbors)
    clf.fit(D_train, y_train)

    y_pred = clf.predict(D_val)
    acc = (y_pred == y_val).mean() * 100
    acc_list.append(acc)

    fold_time = time.time() - t_fold
    fold_times.append(fold_time)
    print(f"Fold {fold} Val Accuracy: {acc:.2f}% | Time: {fold_time:.2f}s")

print("\n===== Final CV Result (SW) =====")
print(f"Avg Acc: {np.mean(acc_list):.2f}% ± {np.std(acc_list):.2f}%")
print(f"Avg Fold Time: {np.mean(fold_times):.2f}s | Total Time: {time.time() - t_all:.2f}s")
